### Import Dependencies

In [17]:
import os
from google import genai
import instructor
from pydantic import BaseModel, Field
import pandas as pd
from sentence_transformers import SentenceTransformer
import cohere

from qdrant_client import QdrantClient
from qdrant_client import models
from qdrant_client.models import Filter, FieldCondition, MatchValue, VectorParams, Distance, SparseVectorParams, Modifier, PayloadSchemaType, PointStruct, Document, Prefetch, FusionQuery

COLLECTION_NAME = "Amazon-items-minilm-l6-v2"
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.5-flash-lite")

### RAG Pipeline

In [2]:
query = "Can I get a tablet?"

In [3]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [4]:
def get_embedding(text, model="sentence-transformers/all-MiniLM-L6-v2"):
    embedding_model = SentenceTransformer(model)
    return embedding_model.encode(text, normalize_embeddings=True).tolist()

In [5]:
def retrieve_data(query, qdrant_client, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01-hybrid-search",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="sentence-transformers-all-MiniLM-L6-v2",
                limit=20
            ),
            Prefetch(
                query=Document(
                    text=query,
                    model="qdrant/bm25"
                ),
                using="bm25",
                limit=20
            )
        ],
        query=FusionQuery(fusion="rrf"),
        limit=k
    )

    retrieved_context_ids = []
    retieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retieved_context.append(result.payload["description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

In [6]:
results = retrieve_data(query, qdrant_client, k=20)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8183.62it/s]


In [7]:
results

{'retrieved_context_ids': ['B09RHF4L45',
  'B0BGHBL86V',
  'B0BN58Z4YX',
  'B0C5QQDN3T',
  'B0B157WDDJ',
  'B0BHVH4D37',
  'B0B6ZZH83Y',
  'B0B58CPFWY',
  'B0B159KDFP',
  'B09WR36NK8',
  'B0BSD3QK7M',
  'B0BCFYCXRH',
  'B0BJ2YWMTP',
  'B0C7DCS2KW',
  'B0BWRZ1HRD',
  'B09W2BMG5N',
  'B09PYV6B4B',
  'B09XQMRYBJ',
  'B09SZNLXJQ',
  'B0B3QVP3W3'],
 'retrieved_context': ['plimpton Android Tablet 10 inch - 8 Core Processor, 4GB RAM 64GB Storage, 1.8M Length USB-C Cable, Android 11, 1920 x 1200 FHD IPS, 13MP Sony Rear Camera, 6000mAh Battery, 2.4G+5G WiFi Tablets PC High Performance Tablet 10.1 inch: Features a faster 1.8GHz Octa-Core CPU and 4GB RAM, you can not only obtain seamless multitasking and hyper-fast content streaming experience, but also can view social media, watch videos, play games, complete homework or finish work. With a 64GB internal storage and support a SD card expanded to 512GB(NOT included), no longer worry about insufficient memory.★★Please note this is a Wi-Fi tablet, 

### Reranking

In [19]:
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Start the notebook kernel inside the project directory.")

load_dotenv(PROJECT_ROOT / ".env", override=True)

True

In [20]:
cohere_api_key = (os.getenv("CO_API_KEY") or "").strip()
if not cohere_api_key:
    raise RuntimeError("Set CO_API_KEY in the project .env file before running reranking.")

cohere_client = cohere.ClientV2(api_key=cohere_api_key)

In [21]:
to_rerank = results["retrieved_context"]

In [22]:
to_rerank

['plimpton Android Tablet 10 inch - 8 Core Processor, 4GB RAM 64GB Storage, 1.8M Length USB-C Cable, Android 11, 1920 x 1200 FHD IPS, 13MP Sony Rear Camera, 6000mAh Battery, 2.4G+5G WiFi Tablets PC High Performance Tablet 10.1 inch: Features a faster 1.8GHz Octa-Core CPU and 4GB RAM, you can not only obtain seamless multitasking and hyper-fast content streaming experience, but also can view social media, watch videos, play games, complete homework or finish work. With a 64GB internal storage and support a SD card expanded to 512GB(NOT included), no longer worry about insufficient memory.★★Please note this is a Wi-Fi tablet, not compatible with SIM card 2022 Latest Android 11 Tablets: Equipped with the latest Android 11 system (GMS certified). Automatic subtitleing, smart reply, notification "bubble", dark theme, eye protection mode etc. It also launches apps 20% faster than Android 10 OS. New control spaces and more privacy features allow you to create a rich user central expression. Y

In [23]:
response = cohere_client.rerank(
    model="rerank-v4.0-pro",
    query=query,
    documents=to_rerank,
    top_n=20
)

In [24]:
response

V2RerankResponse(id='5ccdd0e2-ee3a-49c9-b025-bd4d4166df04', results=[V2RerankResponseResultsItem(index=0, relevance_score=0.86953735, relevant_spans=None), V2RerankResponseResultsItem(index=12, relevance_score=0.8522482, relevant_spans=None), V2RerankResponseResultsItem(index=5, relevance_score=0.8502698, relevant_spans=None), V2RerankResponseResultsItem(index=10, relevance_score=0.84826964, relevant_spans=None), V2RerankResponseResultsItem(index=16, relevance_score=0.8457387, relevant_spans=None), V2RerankResponseResultsItem(index=9, relevance_score=0.84265614, relevant_spans=None), V2RerankResponseResultsItem(index=11, relevance_score=0.8374073, relevant_spans=None), V2RerankResponseResultsItem(index=8, relevance_score=0.83419085, relevant_spans=None), V2RerankResponseResultsItem(index=4, relevance_score=0.8331075, relevant_spans=None), V2RerankResponseResultsItem(index=13, relevance_score=0.8314717, relevant_spans=None), V2RerankResponseResultsItem(index=3, relevance_score=0.8242356

In [25]:
reranked_results = [to_rerank[result.index] for result in response.results]

In [26]:
reranked_results

['plimpton Android Tablet 10 inch - 8 Core Processor, 4GB RAM 64GB Storage, 1.8M Length USB-C Cable, Android 11, 1920 x 1200 FHD IPS, 13MP Sony Rear Camera, 6000mAh Battery, 2.4G+5G WiFi Tablets PC High Performance Tablet 10.1 inch: Features a faster 1.8GHz Octa-Core CPU and 4GB RAM, you can not only obtain seamless multitasking and hyper-fast content streaming experience, but also can view social media, watch videos, play games, complete homework or finish work. With a 64GB internal storage and support a SD card expanded to 512GB(NOT included), no longer worry about insufficient memory.★★Please note this is a Wi-Fi tablet, not compatible with SIM card 2022 Latest Android 11 Tablets: Equipped with the latest Android 11 system (GMS certified). Automatic subtitleing, smart reply, notification "bubble", dark theme, eye protection mode etc. It also launches apps 20% faster than Android 10 OS. New control spaces and more privacy features allow you to create a rich user central expression. Y